In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]

# view
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import folder name
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
print(report_theme)
print(sp_folder_out)
print(folder)

In [ ]:
# Import data by geography
path_in = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)

try: 
    df_tracts = pd.read_excel(os.path.join(path_in, indicator_name + ' Tracts ' + estimate + '.xlsx'), sheet_name = 'Tracts')
    display(df_tracts.head(3))
except Exception as e: print(e)

try: 
    df_counties = pd.read_excel(os.path.join(path_in, indicator_name + ' Counties ' + estimate + '.xlsx'), sheet_name = 'Counties')
    display(df_counties.head(3))
except Exception as e: print(e)

try: 
    df_mpo = pd.read_excel(os.path.join(path_in, indicator_name + ' MPO ' + estimate + '.xlsx'), sheet_name = 'MPO')
    display(df_mpo.head(3))
except Exception as e: print(e)

try: 
    df_msa = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA ' + estimate + '.xlsx'), sheet_name = 'MSA')
    display(df_msa.head(3))
except Exception as e: print(e)

try: 
    df_counties = pd.read_excel(os.path.join(path_in, indicator_name + ' Counties ' +  re.sub('ACS', 'PUMS', estimate) + '.xlsx'), sheet_name = 'PUMA')
    display(df_counties.head(3))
except Exception as e: print(e)

try: 
    df_mpo = pd.read_excel(os.path.join(path_in, indicator_name + ' MPO ' +  re.sub('ACS', 'PUMS', estimate) + '.xlsx'), sheet_name = 'PUMA')
    display(df_mpo.head(3))
except Exception as e: print(e)

In [ ]:
if indicator_name == 'Cost_3':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Vacant']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Vacant']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Vacant']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Vacant']


if indicator_name == 'Cost_5':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Owner occupied']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Owner occupied']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Owner occupied']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Owner occupied']


if indicator_name == 'Health_2':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'No health insurance coverage']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'No health insurance coverage']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'No health insurance coverage']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'No health insurance coverage']

if indicator_name == 'Income_4':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Total Income in the past 12 months below poverty level']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Total Income in the past 12 months below poverty level']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Total Income in the past 12 months below poverty level']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Total Income in the past 12 months below poverty level']

In [ ]:
# remove state FIPS field
# remove "All" category for race/ethnicity
# make sure index is removed
# make sure proportions are now percentages

if geography == 'Tracts':
    df_tracts = df_tracts.drop(['State FIPS', 'County FIPS', 'Tract ID'], axis = 1)
    
    if len(unique(df_tracts.Race_Ethnicity.values)) > 1:
        df_tracts = df_tracts[df_tracts['Race_Ethnicity'] != 'All']
    
    df_tracts = df_tracts.reset_index(drop = True)
    df_tracts.columns = [col.lower() for col in df_tracts.columns]
    df_tracts.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_tracts.columns]
    df_tracts.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_tracts.columns]

if geography == 'Counties':
    df_counties = df_counties.drop(['State FIPS', 'County FIPS'], axis = 1)
    df_mpo    = df_mpo   .drop(['State FIPS'               ], axis = 1)
    
    if len(unique(df_counties.Race_Ethnicity.values)) > 1:
        df_counties = df_counties[df_counties['Race_Ethnicity'] != 'All']
        df_mpo    = df_mpo   [df_mpo   ['Race_Ethnicity'] != 'All']
    
    df_counties.columns = [col.lower() for col in df_counties.columns]
    df_mpo   .columns = [col.lower() for col in df_mpo   .columns]
    
    df_counties.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_counties.columns]
    df_mpo   .columns = [re.sub('[\s+]', '_', col.strip()) for col in df_mpo   .columns]
    
    df_counties.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_counties.columns]
    df_mpo   .columns = [re.sub('\\?'  , '' , col.strip()) for col in df_mpo   .columns]

if geography == 'MSA':
    if len(unique(df_msa.Race_Ethnicity.values)) > 1:
        df_msa = df_msa[df_msa['Race_Ethnicity'] != 'All']
    else:
        df_msa = df_msa.copy()
        
    df_msa  = df_msa.reset_index(drop = True)
    df_msa.columns = [x.lower() for x in df_msa.columns]
    df_msa.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_msa.columns]
    df_msa.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_msa.columns]

if geography == 'PUMA':
    df_puma.columns = [re.sub('_desc', '', col) for col in df_puma.columns]
    df_puma = df_puma.rename(columns = {'Year':'year', 'state': 'State FIPS'})
    

In [ ]:

if geography == 'Tracts':
    display(df_tracts.head(3))

if geography == 'Counties':
    display(df_counties.head(3), df_mpo.head(3))

if geography == 'MSA':
    display(df_msa.head(3))

if geography == 'PUMA':
    display(df_counties.head(3), df_mpo.head(3))
    

In [ ]:
# Set output name for .csv files

if geography == 'Tracts':
    name_output_tracts_csv  = [indicator_name, '_Tracts_' , estimate, '.csv']
    name_output_tracts_csv  = "".join(name_output_tracts_csv)
    
if geography == 'Counties':
    name_output_counties_csv = [indicator_name, '_Counties_', estimate, '.csv']
    name_output_MPO_csv      = [indicator_name, '_MPO_'     , estimate, '.csv']
    name_output_counties_csv = "".join(name_output_counties_csv)
    name_output_MPO_csv      = "".join(name_output_MPO_csv     )
    
if geography == 'MSA':
    name_output_MSA_csv = [indicator_name, '_MSA_', estimate, '.csv']
    name_output_MSA_csv = "".join(name_output_MSA_csv)

if geography == 'PUMA':
    name_output_PUMA_csv = [indicator_name, '_PUMA_', estimate, '.csv']
    name_output_PUMA_csv = "".join(name_output_PUMA_csv)
    

In [ ]:
# Set file path for exporting
path_out_csv  = os.path.join(path_agol, indicator_name)
print('CSV files exported here: ' + path_out_csv )

if geography == 'Tracts':
    df_tracts.to_csv(os.path.join(path_out_csv, name_output_tracts_csv), index = False)

if geography == 'Counties':
    df_counties.to_csv(os.path.join(path_out_csv, name_output_counties_csv), index = False)
    df_mpo     .to_csv(os.path.join(path_out_csv, name_output_MPO_csv     ), index = False)

if geography == 'MSA':
    df_msa.to_csv(os.path.join(path_out_csv, name_output_MSA_csv), index = False)

if geography == 'PUMA':
    df_puma.to_csv(os.path.join(path_out_csv, name_output_PUMA_csv), index = False)

print('')
print("Successfully exported")